<a href="https://colab.research.google.com/github/Shambalam/RL-for-game-AI/blob/main/Copy_of_npc_project_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Style-Conditioned NPC Behavior via Deep Reinforcement Learning
### Tactical Grid-World Environment

**Before running:** Go to `Runtime → Change runtime type → T4 GPU`

---
**Project structure (all in this notebook):**
1. Install dependencies
2. Grid environment
3. Style-conditioned policy architecture
4. Training (all 4 styles)
5. Evaluation + comparison plots

## 1. Install Dependencies

In [ ]:
!pip install stable-baselines3[extra] gymnasium torch matplotlib -q

import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('WARNING: No GPU detected. Go to Runtime > Change runtime type > T4 GPU')

## 2. Grid Environment
A procedurally generated 16x16 tactical grid-world. The NPC is the learning agent. The player runs a scripted chase-and-shoot policy.

In [ ]:
"""
grid_env.py  —  2D Tactical Grid-World Environment
====================================================
NPC is the learning agent. Player runs a scripted chase-and-shoot policy.

Grid channels:
  0 = obstacles (impassable walls)
  1 = cover tiles (reduce exposure)
  2 = player position
  3 = NPC position

Observation:
  'grid'    : (4, view, view)  local occupancy centred on NPC
  'scalars' : (4,)  [npc_hp, npc_ammo, dist_to_player, angle_to_player]
  'style'   : (4,)  one-hot style vector

Actions (Discrete 6):
  0=up  1=down  2=left  3=right  4=stay  5=shoot

Styles:
  0 = Aggressive   close in, deal damage, stay in the player's face
  1 = Defensive    peek from cover, shoot, return to cover
  2 = Flanker      approach outside player FOV, attack from the side/rear
  3 = Sniper       maintain distance, acquire LOS, take long-range shots
"""

import numpy as np
import random
import gymnasium as gym
from gymnasium import spaces

STYLE_NAMES = {0: 'aggressive', 1: 'defensive', 2: 'flanker', 3: 'sniper'}
N_STYLES    = 4
N_ACTIONS   = 6

MOVE_DELTAS = {
    0: np.array([-1,  0]),
    1: np.array([ 1,  0]),
    2: np.array([ 0, -1]),
    3: np.array([ 0,  1]),
}


class TacticalGridEnv(gym.Env):
    metadata = {'render_modes': ['ansi']}

    def __init__(self, grid_size=16, view_radius=5, style=0, max_steps=300):
        super().__init__()
        self.grid_size   = grid_size
        self.view_radius = view_radius
        self.style       = style
        self.max_steps   = max_steps

        v = 2 * view_radius + 1
        self.observation_space = spaces.Dict({
            'grid':    spaces.Box(0., 1., shape=(4, v, v),     dtype=np.float32),
            'scalars': spaces.Box(-1., 1., shape=(4,),         dtype=np.float32),
            'style':   spaces.Box(0., 1., shape=(N_STYLES,),   dtype=np.float32),
        })
        self.action_space = spaces.Discrete(N_ACTIONS)

        # ── state (initialised properly in reset) ──────────────────────
        self.base_grid    = np.zeros((2, grid_size, grid_size), dtype=np.float32)
        self.player_pos   = np.zeros(2, dtype=np.int32)
        self.npc_pos      = np.zeros(2, dtype=np.int32)
        self.player_hp    = 100
        self.npc_hp       = 100
        self.npc_ammo     = 20
        self.player_ammo  = 20
        self.step_count   = 0
        # Player facing direction — updated from movement each step
        self.player_facing = np.array([0., 1.], dtype=np.float32)
        # Defensive peek tracker — was the NPC in cover last step?
        self.was_in_cover  = False

    # ================================================================== #
    #  Map generation
    # ================================================================== #
    def _generate_map(self):
        """Return (2, G, G): channel 0 = obstacles, channel 1 = cover."""
        g = np.zeros((2, self.grid_size, self.grid_size), dtype=np.float32)

        # Scatter obstacles — avoid the border row/col so agents don't get trapped
        n_obs = int(self.grid_size ** 2 * 0.15)
        for _ in range(n_obs):
            x = random.randint(1, self.grid_size - 2)
            y = random.randint(1, self.grid_size - 2)
            g[0, x, y] = 1.

        # Scatter cover on free tiles
        n_cov = int(self.grid_size ** 2 * 0.10)
        for _ in range(n_cov):
            x = random.randint(0, self.grid_size - 1)
            y = random.randint(0, self.grid_size - 1)
            if g[0, x, y] == 0:
                g[1, x, y] = 1.

        return g

    def _free_pos(self):
        """Random position not occupied by an obstacle."""
        while True:
            x = random.randint(0, self.grid_size - 1)
            y = random.randint(0, self.grid_size - 1)
            if self.base_grid[0, x, y] == 0:
                return np.array([x, y], dtype=np.int32)

    # ================================================================== #
    #  Reset
    # ================================================================== #
    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        if seed is not None:
            random.seed(seed)
            np.random.seed(seed)

        self.base_grid  = self._generate_map()
        self.player_pos = self._free_pos()
        self.npc_pos    = self._free_pos()

        # Force a minimum starting separation
        for _ in range(100):
            if np.linalg.norm(self.player_pos - self.npc_pos) >= self.grid_size // 3:
                break
            self.npc_pos = self._free_pos()

        self.player_hp    = 100
        self.npc_hp       = 100
        self.npc_ammo     = 20
        self.player_ammo  = 20
        self.step_count   = 0
        self.player_facing = np.array([0., 1.], dtype=np.float32)
        self.was_in_cover  = False

        return self._obs(), {}

    # ================================================================== #
    #  Observation
    # ================================================================== #
    def _local_grid(self, center):
        r, s = self.view_radius, 2 * self.view_radius + 1
        out = np.zeros((4, s, s), dtype=np.float32)
        for i in range(s):
            for j in range(s):
                gx = center[0] - r + i
                gy = center[1] - r + j
                if 0 <= gx < self.grid_size and 0 <= gy < self.grid_size:
                    out[0, i, j] = self.base_grid[0, gx, gy]   # obstacle
                    out[1, i, j] = self.base_grid[1, gx, gy]   # cover
                    if gx == self.player_pos[0] and gy == self.player_pos[1]:
                        out[2, i, j] = 1.                       # player
                    if gx == self.npc_pos[0] and gy == self.npc_pos[1]:
                        out[3, i, j] = 1.                       # npc
                else:
                    out[0, i, j] = 1.                           # out-of-bounds = wall
        return out

    def _obs(self):
        diff  = (self.player_pos - self.npc_pos).astype(np.float32)
        dist  = float(np.linalg.norm(diff)) / self.grid_size
        angle = float(np.arctan2(diff[1], diff[0])) / np.pi

        scalars  = np.array([self.npc_hp / 100., self.npc_ammo / 20., dist, angle],
                            dtype=np.float32)
        style_oh = np.zeros(N_STYLES, dtype=np.float32)
        style_oh[self.style] = 1.

        return {
            'grid':    self._local_grid(self.npc_pos),
            'scalars': scalars,
            'style':   style_oh,
        }

    # ================================================================== #
    #  Movement / combat helpers
    # ================================================================== #
    def _move(self, pos, action):
        if action not in MOVE_DELTAS:
            return pos.copy()
        new = pos + MOVE_DELTAS[action]
        if (0 <= new[0] < self.grid_size and
                0 <= new[1] < self.grid_size and
                self.base_grid[0, new[0], new[1]] == 0):
            return new
        return pos.copy()

    def _in_cover(self, pos):
        return bool(self.base_grid[1, pos[0], pos[1]] == 1)

    def _has_los(self, a, b):
        """Bresenham line-of-sight — returns False if any obstacle is in the way."""
        x0, y0 = int(a[0]), int(a[1])
        x1, y1 = int(b[0]), int(b[1])
        dx, dy = abs(x1 - x0), abs(y1 - y0)
        sx = 1 if x0 < x1 else -1
        sy = 1 if y0 < y1 else -1
        err = dx - dy
        cx, cy = x0, y0
        while not (cx == x1 and cy == y1):
            if self.base_grid[0, cx, cy] == 1:
                return False
            e2 = 2 * err
            if e2 > -dy: err -= dy; cx += sx
            if e2 <  dx: err += dx; cy += sy
        return True

    def _shoot(self, shooter, target, ammo_attr, hp_attr):
        """Fire if ammo and LOS available. Returns True on hit."""
        if getattr(self, ammo_attr) <= 0:
            return False
        if not self._has_los(shooter, target):
            return False
        setattr(self, ammo_attr, getattr(self, ammo_attr) - 1)
        dist      = np.linalg.norm(shooter - target)
        hit_chance = max(0.15, 1. - dist / self.grid_size)
        if random.random() < hit_chance:
            setattr(self, hp_attr, getattr(self, hp_attr) - 10)
            return True
        return False

    # ================================================================== #
    #  Scripted player policy
    # ================================================================== #
    def _step_player(self):
        """
        Simple scripted behaviour:
          - If within range 5, shoot at the NPC.
          - Otherwise, move one step closer.
        Also updates player_facing from movement or aim direction.
        """
        dist = np.linalg.norm(self.player_pos - self.npc_pos)

        if dist <= 5:
            # Update facing toward NPC even when stationary
            face = (self.npc_pos - self.player_pos).astype(np.float32)
            norm = np.linalg.norm(face) + 1e-6
            self.player_facing = face / norm
            self._shoot(self.player_pos, self.npc_pos, 'player_ammo', 'npc_hp')
        else:
            diff   = self.npc_pos - self.player_pos
            action = (
                (1 if diff[0] > 0 else 0)
                if abs(diff[0]) >= abs(diff[1])
                else (3 if diff[1] > 0 else 2)
            )
            old_pos          = self.player_pos.copy()
            self.player_pos  = self._move(self.player_pos, action)
            moved            = (self.player_pos - old_pos).astype(np.float32)
            if np.linalg.norm(moved) > 0:
                self.player_facing = moved / np.linalg.norm(moved)
            else:
                # Blocked — face toward NPC anyway
                face = (self.npc_pos - self.player_pos).astype(np.float32)
                norm = np.linalg.norm(face) + 1e-6
                self.player_facing = face / norm

    # ================================================================== #
    #  Flanker helpers
    # ================================================================== #
    def _lateral_score(self):
        """
        How lateral is the NPC's position relative to the player?
        Uses the NPC-to-player vector projected onto the grid axes.

          1.0 = NPC is directly to the side of the player (pure lateral)
          0.0 = NPC is directly in front or behind the player

        This is robust regardless of where the player is facing.
        """
        diff = (self.npc_pos - self.player_pos).astype(np.float32)
        norm = np.linalg.norm(diff) + 1e-6
        # diff[0] is the row axis (forward/back), diff[1] is column (side)
        return 1.0 - abs(diff[0] / norm)

    def _in_player_fov(self, fov_half_deg=90.0):
        """
        Returns True if the NPC is inside the player's FOV cone.
        player_facing is updated from movement so it reflects actual
        facing direction rather than always pointing at the NPC.
        A true flank means this returns False.
        """
        to_npc = (self.npc_pos - self.player_pos).astype(np.float32)
        norm   = np.linalg.norm(to_npc) + 1e-6
        dot    = float(np.dot(self.player_facing, to_npc / norm))
        return dot >= np.cos(np.radians(fov_half_deg))

    # ================================================================== #
    #  Style-specific reward
    # ================================================================== #
    def _reward(self, action, prev_dist, new_dist, hit_player, took_damage):
        r = 0.

        # ── Aggressive ────────────────────────────────────────────────
        if self.style == 0:
            # Close in and deal damage. Stronger signal, close-range bonus.
            r += (prev_dist - new_dist) * 6.0   # stronger closing reward
            if new_dist < 0.35:
                r += 0.8                         # bonus for staying in close range
            if hit_player:
                r += 8.0                         # increased hit reward
            if action == 4:
                r -= 1.2                         # stronger idle penalty

        # ── Defensive ─────────────────────────────────────────────────
        elif self.style == 1:
            currently_in_cover = self._in_cover(self.npc_pos)
            has_los            = self._has_los(self.npc_pos, self.player_pos)

            r += 0.05                            # survive each step
            if currently_in_cover:
                r += 0.4                         # reward sitting in cover

            if took_damage:
                r -= 2.0                         # getting hit is bad

            if hit_player:
                r += 4.0                         # dealing damage is good

            # Peek-shoot mechanic ─────────────────────────────────────
            # Step A: left cover, have LOS, and chose to shoot → reward the peek shot
            if not currently_in_cover and has_los and action == 5:
                r += 1.5

            # Step B: back in cover after being outside → reward the retreat
            if currently_in_cover and not self.was_in_cover:
                r += 1.0

            # Track cover state for next step
            self.was_in_cover = currently_in_cover

        # ── Flanker ───────────────────────────────────────────────────
        elif self.style == 2:
            lateral = self._lateral_score()      # 0 = front/back, 1 = pure side
            in_fov  = self._in_player_fov(90.)   # True = player can currently see us

            # Approach phase (too far to engage)
            if new_dist > 0.45:
                if not in_fov:
                    r += (prev_dist - new_dist) * 3.0  # fast approach while unseen
                else:
                    r += (prev_dist - new_dist) * 0.5  # slow, discouraged frontal approach

            # Positioning phase (within engagement range)
            if new_dist <= 0.45:
                r += lateral * 1.5               # reward lateral position
                if not in_fov:
                    r += 0.5                     # extra bonus for being unseen

            # Stealth movement: cover + outside FOV
            if action in MOVE_DELTAS and self._in_cover(self.npc_pos) and not in_fov:
                r += 0.3                         # sneaking through cover

            # Strike rewards — stacked bonuses for quality flanks
            if hit_player:
                r += 5.0                         # base hit
                if not in_fov:
                    r += 3.0                     # hit from outside FOV
                if lateral > 0.5:
                    r += 2.0                     # genuinely lateral hit

            # Penalise walking straight at a player who sees you
            if in_fov and lateral < 0.2:
                r -= 0.4                         # frontal charge = wrong style

            if action == 4:
                r -= 0.5                         # flanker must keep moving

        # ── Sniper ────────────────────────────────────────────────────
        elif self.style == 3:
            has_los = self._has_los(self.npc_pos, self.player_pos)

            # Tiered distance reward — just positive tiers, no penalties
            # Avoids reward drain accumulating over 300 steps
            if new_dist > 0.6:
                r += 0.5                         # long range = good
            elif new_dist > 0.45:
                r += 0.15                        # medium range = acceptable

            # Reward actively increasing distance when player is close
            # new_dist > prev_dist means NPC moved away from player
            if new_dist > prev_dist and prev_dist < 0.5:
                r += (new_dist - prev_dist) * 4.0   # reward backing away

            # LOS at range is valuable
            if has_los and new_dist > 0.4:
                r += 0.3

            # Big payout for landing a shot from range
            if hit_player and new_dist > 0.4:
                r += 10.0

            # Movement allowed freely — sniper needs to kite and reposition
            if action in MOVE_DELTAS and has_los and new_dist > 0.5:
                r -= 0.2                         # only penalise unnecessary movement when safe

        # ── Global terms (all styles) ──────────────────────────────────
        r -= 0.01                                # small step cost to discourage idling

        if self.player_hp <= 0:
            r += 20.0                            # win bonus
        if self.npc_hp <= 0:
            r -= 20.0                            # death penalty

        return float(r)

    # ================================================================== #
    #  Step
    # ================================================================== #
    def step(self, action):
        self.step_count  += 1
        prev_dist         = np.linalg.norm(self.player_pos - self.npc_pos) / self.grid_size
        npc_hp_before     = self.npc_hp
        hit_player        = False

        # NPC action
        if action == 5:
            hit_player = self._shoot(self.npc_pos, self.player_pos, 'npc_ammo', 'player_hp')
        elif action in MOVE_DELTAS:
            self.npc_pos = self._move(self.npc_pos, action)
        # action == 4 → stay

        # Player acts after NPC
        self._step_player()

        took_damage = self.npc_hp < npc_hp_before
        new_dist    = np.linalg.norm(self.player_pos - self.npc_pos) / self.grid_size

        reward     = self._reward(action, prev_dist, new_dist, hit_player, took_damage)
        terminated = self.player_hp <= 0 or self.npc_hp <= 0
        truncated  = self.step_count >= self.max_steps

        return self._obs(), reward, terminated, truncated, {}

    # ================================================================== #
    #  Render (text)
    # ================================================================== #
    def render(self):
        d = [['.' for _ in range(self.grid_size)] for _ in range(self.grid_size)]
        for x in range(self.grid_size):
            for y in range(self.grid_size):
                if   self.base_grid[0, x, y] == 1: d[x][y] = '#'
                elif self.base_grid[1, x, y] == 1: d[x][y] = 'c'
        d[self.player_pos[0]][self.player_pos[1]] = 'P'
        d[self.npc_pos[0]][self.npc_pos[1]]       = 'N'
        print(f"\nStyle: {STYLE_NAMES[self.style]}  |  Step: {self.step_count}")
        print(f"Player HP: {self.player_hp}  NPC HP: {self.npc_hp}  NPC Ammo: {self.npc_ammo}")
        print(f"Player facing: ({self.player_facing[0]:.2f}, {self.player_facing[1]:.2f})")
        for row in d:
            print(' '.join(row))

## 3. Style-Conditioned Policy Architecture
```
grid (4×11×11) → CNN ─────────────────────┐
scalars (4,)   → MLP (32) ────────────────┤→ Concat → Fusion MLP (128) → Actor/Critic
style (4,)     → Linear Embedding (16) ───┘
```

In [ ]:
import torch
import torch.nn as nn
from gymnasium import spaces
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor

class StyleConditionedExtractor(BaseFeaturesExtractor):
    def __init__(self, observation_space: spaces.Dict, embedding_dim=16, features_dim=128):
        super().__init__(observation_space, features_dim=features_dim)

        grid_shape = observation_space['grid'].shape    # (4, view, view)
        n_scalars  = observation_space['scalars'].shape[0]

        # CNN for occupancy grid
        self.cnn = nn.Sequential(
            nn.Conv2d(grid_shape[0], 16, kernel_size=3, padding=1), nn.ReLU(),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, stride=2, padding=1),  nn.ReLU(),
            nn.Flatten(),
        )
        with torch.no_grad():
            cnn_out = self.cnn(torch.zeros(1, *grid_shape)).shape[1]

        # MLP for scalar features
        self.scalar_net = nn.Sequential(
            nn.Linear(n_scalars, 32), nn.ReLU(),
            nn.Linear(32, 32),        nn.ReLU(),
        )

        # Style embedding
        self.style_embed = nn.Sequential(
            nn.Linear(N_STYLES, embedding_dim), nn.ReLU(),
        )

        # Fusion MLP
        self.fusion = nn.Sequential(
            nn.Linear(cnn_out + 32 + embedding_dim, features_dim), nn.ReLU(),
            nn.Linear(features_dim, features_dim),                 nn.ReLU(),
        )

    def forward(self, obs):
        g = self.cnn(obs['grid'])
        s = self.scalar_net(obs['scalars'])
        e = self.style_embed(obs['style'])
        return self.fusion(torch.cat([g, s, e], dim=1))

print('Architecture defined OK')

## 4. Training
Trains all 4 styles sequentially. Each gets its own PPO model with separate style-specific rewards.

**Estimated time on T4 GPU:** ~5–10 min per style at 300k steps.

In [ ]:
import os
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import VecMonitor

# ── Config ────────────────────────────────────────────────────────────
TIMESTEPS_PER_STYLE = 500_000   # increase to 1_000_000 for better results
N_ENVS              = 8         # parallel environments
SAVE_DIR            = '/content/models'
os.makedirs(SAVE_DIR, exist_ok=True)

policy_kwargs = dict(
    features_extractor_class=StyleConditionedExtractor,
    features_extractor_kwargs=dict(embedding_dim=16, features_dim=128),
    net_arch=dict(pi=[128, 64], vf=[128, 64]),
)

trained_models = {}

for style_id, style_name in STYLE_NAMES.items():
    print(f"\n{'='*55}")
    print(f"  Training: {style_name.upper()}  ({TIMESTEPS_PER_STYLE:,} steps)")
    print(f"{'='*55}")

    train_env = VecMonitor(make_vec_env(
        lambda sid=style_id: TacticalGridEnv(style=sid),
        n_envs=N_ENVS
    ))

    model = PPO(
        policy='MultiInputPolicy',
        env=train_env,
        learning_rate=3e-4,
        n_steps=512,
        batch_size=256,
        n_epochs=10,
        gamma=0.99,
        gae_lambda=0.95,
        clip_range=0.2,
        ent_coef=0.01,
        vf_coef=0.5,
        max_grad_norm=0.5,
        policy_kwargs=policy_kwargs,
        verbose=1,
        device='cuda' if torch.cuda.is_available() else 'cpu',
    )

    model.learn(total_timesteps=TIMESTEPS_PER_STYLE, progress_bar=True)

    save_path = os.path.join(SAVE_DIR, f'{style_name}_final')
    model.save(save_path)
    trained_models[style_id] = model
    train_env.close()
    print(f'  Saved → {save_path}.zip')

print('\nAll styles trained.')

## 5. Evaluation & Comparison Plots

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

STYLE_COLORS = {
    'aggressive': '#e74c3c',
    'defensive':  '#2ecc71',
    'flanker':    '#3498db',
    'sniper':     '#9b59b6',
}
N_EVAL_EPISODES = 30

all_results = []

for style_id, style_name in STYLE_NAMES.items():
    model = trained_models[style_id]
    env   = TacticalGridEnv(style=style_id)

    rewards, wins, dists, dmg_dealt, dmg_taken, steps_list = [], [], [], [], [], []

    for ep in range(N_EVAL_EPISODES):
        obs, _ = env.reset()
        done, ep_r, ep_d = False, 0., []
        prev_php, prev_nhp = 100, 100
        ep_dd, ep_dt = 0, 0

        while not done:
            action, _ = model.predict(obs, deterministic=True)
            obs, r, terminated, truncated, _ = env.step(int(action))
            ep_r   += r
            done    = terminated or truncated
            ep_d.append(np.linalg.norm(env.player_pos - env.npc_pos))
            ep_dd  += prev_php - env.player_hp
            ep_dt  += prev_nhp - env.npc_hp
            prev_php, prev_nhp = env.player_hp, env.npc_hp

        rewards.append(ep_r)
        wins.append(1 if env.player_hp <= 0 else 0)
        dists.append(np.mean(ep_d))
        dmg_dealt.append(ep_dd)
        dmg_taken.append(ep_dt)
        steps_list.append(env.step_count)

    result = {
        'style':          style_name,
        'mean_reward':    np.mean(rewards),
        'win_rate':       np.mean(wins),
        'avg_distance':   np.mean(dists),
        'avg_dmg_dealt':  np.mean(dmg_dealt),
        'avg_dmg_taken':  np.mean(dmg_taken),
    }
    all_results.append(result)
    print(f"{style_name.upper():12} | reward={result['mean_reward']:7.2f} | "
          f"win={result['win_rate']:.2f} | dist={result['avg_distance']:.1f} | "
          f"dealt={result['avg_dmg_dealt']:.1f} | taken={result['avg_dmg_taken']:.1f}")

print('\nEvaluation complete.')

In [ ]:
# ── Comparison bar chart ───────────────────────────────────────────────
styles = [r['style'] for r in all_results]
colors = [STYLE_COLORS[s] for s in styles]

metrics = {
    'Mean Reward':     [r['mean_reward']   for r in all_results],
    'Win Rate':        [r['win_rate']      for r in all_results],
    'Avg Distance':    [r['avg_distance']  for r in all_results],
    'Damage Dealt':    [r['avg_dmg_dealt'] for r in all_results],
    'Damage Taken':    [r['avg_dmg_taken'] for r in all_results],
}

fig, axes = plt.subplots(1, len(metrics), figsize=(20, 5))
fig.suptitle('Style-Conditioned NPC — Behavior Comparison Across Styles',
             fontsize=14, fontweight='bold', y=1.02)

for ax, (name, vals) in zip(axes, metrics.items()):
    bars = ax.bar(styles, vals, color=colors, edgecolor='black', linewidth=0.7)
    ax.set_title(name, fontsize=11)
    ax.set_xticks(range(len(styles)))
    ax.set_xticklabels([s.capitalize() for s in styles], rotation=20, ha='right')
    ax.bar_label(bars, fmt='%.2f', padding=3, fontsize=9)
    ax.grid(axis='y', linestyle='--', alpha=0.4)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

patches = [mpatches.Patch(color=STYLE_COLORS[s], label=s.capitalize()) for s in styles]
fig.legend(handles=patches, loc='lower center', ncol=4, fontsize=10,
           frameon=False, bbox_to_anchor=(0.5, -0.08))

plt.tight_layout()
plt.savefig('/content/style_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved to /content/style_comparison.png')

In [ ]:
# ── Watch a single episode in text ────────────────────────────────────
# Change WATCH_STYLE to see any of the four behaviors
WATCH_STYLE = 0   # 0=aggressive 1=defensive 2=flanker 3=sniper

model = trained_models[WATCH_STYLE]
env   = TacticalGridEnv(style=WATCH_STYLE)
obs, _ = env.reset()
done = False

print(f"Watching: {STYLE_NAMES[WATCH_STYLE].upper()}\n")
step = 0
while not done and step < 30:
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, _ = env.step(int(action))
    done = terminated or truncated
    if step % 5 == 0 or done:
        env.render()
        print(f'  reward this step: {reward:.3f}\n')
    step += 1

In [ ]:
# ── Download models to your local machine ─────────────────────────────
from google.colab import files
import zipfile, os

zip_path = '/content/trained_models.zip'
with zipfile.ZipFile(zip_path, 'w') as zf:
    for style_name in STYLE_NAMES.values():
        p = f'/content/models/{style_name}_final.zip'
        if os.path.exists(p):
            zf.write(p, arcname=f'{style_name}_final.zip')

files.download(zip_path)
print('Download started.')